# 📊 Análisis Exploratorio de Datos — Evaluaciones Docentes
### Reto *Analítica Educativa* · Universidad del Rosario
**Samsung Innovation Campus 2025 — Proyecto Capstone**

| | |
|---|---|
| **Equipo** | Tech Tailors |
| **Integrantes** | Juan David Ríos · Natalia Remolina · Giovanni Balza |
| **Historias cubiertas** | `SCRUM-5` Gráficas del Análisis Exploratorio · `SCRUM-8` Determinación de features más significativas para la variable objetivo |
| **Dataset** | `evaluaciones_docentes.csv` (3,000 evaluaciones · 50 docentes · 8 semestres) |

---

### 🎯 Premisa del reto

> *Diseñar e implementar una solución avanzada de análisis de evaluaciones docentes que evolucione el sistema actual basado en reglas hacia un modelo de aprendizaje automático, manteniendo compatibilidad con los datos y procesos existentes. El reto consiste en pasar de un análisis descriptivo a uno predictivo y accionable, que permita anticipar tendencias en el desempeño docente y generar recomendaciones de mejora continua.*

### Objetivos de este notebook

1. **Diagnosticar** la calidad e integridad del dataset (fuente de verdad para todo el pipeline).
2. **Caracterizar** las distribuciones de puntajes, grupos y la variable objetivo `tendencia_desempeno`.
3. **Explorar** relaciones bivariadas, evolución temporal, diferencias por asignatura y señal textual de los comentarios.
4. **Cuantificar** (SCRUM-8) qué variables aportan mayor poder predictivo sobre `tendencia_desempeno`, mediante pruebas estadísticas, información mutua e importancia de permutación.
5. **Consolidar** insights accionables que definan el diseño del modelo de la siguiente fase.

### Tabla de contenido

1. Configuración del entorno
2. Carga y validación del dataset
3. Diagnóstico de calidad de datos
4. Análisis univariado
5. Análisis bivariado y correlaciones
6. Análisis temporal
7. Análisis por asignatura
8. Análisis de comentarios (señal textual)
9. Perfil por docente
10. SCRUM-8 · Significancia de features para la variable objetivo
11. Conclusiones, limitaciones y próximos pasos


## 1 · Configuración del entorno

Fijamos una **identidad visual única** para todo el análisis (paleta categórica validada para
daltonismo, colores semánticos fijos para la variable objetivo) y una semilla aleatoria para
garantizar **reproducibilidad**. Toda gráfica del notebook usa estas convenciones:

* 🟢 `Mejora` · 🔵 `Estable` · 🔴 `En riesgo` — el color sigue a la clase en **todas** las gráficas.
* Escala secuencial de azules para magnitudes (mapas de calor).
* Paleta categórica de orden fijo para asignaturas.

In [ ]:
# === Librerías ===
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# === Identidad visual del proyecto ===
PALETA_CAT = ['#2a78d6', '#1baf7a', '#eda100', '#008300',
              '#4a3aa7', '#e34948', '#e87ba4', '#eb6834']      # categóricas, orden fijo
COLOR_TENDENCIA = {'Mejora': '#0ca30c', 'Estable': '#2a78d6', 'En riesgo': '#d03b3b'}
COLOR_SENTIMIENTO = {'positivo': '#0ca30c', 'neutro': '#898781', 'negativo': '#d03b3b'}
ORDEN_TENDENCIA = ['Mejora', 'Estable', 'En riesgo']
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#c3c2b7', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.6,
    'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': INK,
    'axes.labelsize': 10, 'axes.labelcolor': INK2,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'legend.fontsize': 9,
})
sns.set_palette(PALETA_CAT)
print('Entorno configurado ✔  |  semilla =', RANDOM_STATE)

## 2 · Carga y validación del dataset

**Decisión de gobierno de datos** (auditoría previa de archivos fuente):

| Archivo | Veredicto |
|---|---|
| `evaluaciones_docentes.csv` | ✅ **Fuente de verdad** |
| `Evaluaciones docentes.csv` | ⚠️ Duplicado byte a byte del anterior (mismo hash MD5) — se descarta |
| `Evaluaciones docentes.xlsx` | ❌ Columna `semestre` corrompida por auto-conversión de Excel (`2020-2` → *feb-2020*) — se descarta |

La celda siguiente funciona tanto en **Google Colab** (solicita subir el archivo) como en
entorno local (lo lee del directorio de trabajo).

In [ ]:
CSV_PATH = Path('evaluaciones_docentes.csv')

if not CSV_PATH.exists():
    try:                                    # --- Entorno Google Colab ---
        from google.colab import files
        print('⬆ Sube el archivo evaluaciones_docentes.csv')
        uploaded = files.upload()
        CSV_PATH = Path(next(iter(uploaded)))
    except ImportError:                     # --- Entorno local ---
        raise FileNotFoundError('Coloca evaluaciones_docentes.csv en el directorio de trabajo.')

df = pd.read_csv(CSV_PATH)
COLS_PUNTAJE = ['puntaje_claridad', 'puntaje_metodologia', 'puntaje_evaluacion']

print(f'Dimensiones: {df.shape[0]:,} evaluaciones × {df.shape[1]} variables')
df.head()

### Diccionario de datos

| Variable | Tipo | Descripción |
|---|---|---|
| `id_docente` | categórica (50) | Identificador anonimizado del docente |
| `asignatura` | categórica (7) | Materia evaluada |
| `semestre` | ordinal (8) | Periodo académico, `2020-1` … `2023-2` |
| `numero_estudiantes` | numérica | Tamaño del grupo que evaluó (15–45) |
| `puntaje_claridad` | numérica [1–5] | Claridad al explicar |
| `puntaje_metodologia` | numérica [1–5] | Calidad metodológica |
| `puntaje_evaluacion` | numérica [1–5] | Justicia/pertinencia de la evaluación |
| `comentario` | texto | Comentario cualitativo del grupo |
| `tendencia_desempeno` | **objetivo** (3 clases) | `Mejora` · `Estable` · `En riesgo` |

> Cada fila es la **evaluación agregada de un grupo** de estudiantes a un docente en una
> asignatura y semestre — no la respuesta de un estudiante individual.

## 3 · Diagnóstico de calidad de datos

Antes de extraer conclusiones verificamos: completitud (nulos), unicidad (duplicados),
validez (rangos y formatos) y consistencia de categorías.

In [ ]:
# --- Completitud, tipos y cardinalidad ---
calidad = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().mean() * 100).round(2),
    'valores únicos': df.nunique(),
})
print(f'Filas duplicadas exactas: {df.duplicated().sum()}')
calidad

In [ ]:
# --- Reglas de validez del negocio ---
checks = {
    'Puntajes dentro de [1, 5]':
        df[COLS_PUNTAJE].apply(lambda s: s.between(1, 5)).all().all(),
    'numero_estudiantes > 0':
        (df['numero_estudiantes'] > 0).all(),
    'Formato de semestre AAAA-{1,2}':
        df['semestre'].str.fullmatch(r'\d{4}-[12]').all(),
    'Objetivo con exactamente 3 clases válidas':
        set(df['tendencia_desempeno'].unique()) == {'Mejora', 'Estable', 'En riesgo'},
    'Sin comentarios vacíos':
        df['comentario'].str.strip().ne('').all(),
}
for regla, ok in checks.items():
    print(f"{'✅' if ok else '❌'}  {regla}")

> 📌 **Hallazgos de calidad**
> * Dataset **íntegro**: 0 nulos, 0 duplicados, 0 valores fuera de rango. No se requiere imputación ni limpieza estructural.
> * Cobertura completa: los 50 docentes tienen registros en los 8 semestres (400 combinaciones docente-semestre).
> * Señales de **origen sintético** que se documentan como limitación (§ 11): solo 20 comentarios únicos, todos los docentes dictan las 7 asignaturas y presentan las 3 clases de tendencia.

## 4 · Análisis univariado

### 4.1 · Variable objetivo: `tendencia_desempeno`

La primera pregunta de cualquier problema de clasificación: **¿qué tan balanceadas están las clases?**
De esto dependen la métrica de evaluación y la estrategia de entrenamiento.

In [ ]:
conteo = df['tendencia_desempeno'].value_counts().reindex(ORDEN_TENDENCIA)

fig, ax = plt.subplots(figsize=(8, 3))
orden_plot = conteo.index[::-1]
bars = ax.barh(orden_plot, conteo[orden_plot],
               color=[COLOR_TENDENCIA[t] for t in orden_plot], height=0.62)
for b, t in zip(bars, orden_plot):
    v = conteo[t]
    ax.text(v + 25, b.get_y() + b.get_height()/2, f'{v:,}  ({v/len(df):.1%})',
            va='center', fontsize=10, color=INK2)
ax.set_xlim(0, conteo.max() * 1.3)
ax.set_title('Balance de clases de la variable objetivo')
ax.set_xlabel('Número de evaluaciones')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

> 📌 **Insight 1 — Desbalance moderado de clases.** `Estable` concentra el 50% de los casos,
> mientras la clase **más crítica para el negocio (`En riesgo`) es la minoritaria (~20%)**.
> Implicaciones para el modelado:
> * La métrica principal no puede ser *accuracy*: usaremos **macro-F1** y **recall de `En riesgo`**
>   (el costo de no detectar un docente en riesgo es mayor que una falsa alarma).
> * Aplicar `class_weight='balanced'` o sobremuestreo en el entrenamiento.

### 4.2 · Distribución de las variables numéricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
variables = COLS_PUNTAJE + ['numero_estudiantes']
titulos = ['Claridad', 'Metodología', 'Evaluación', 'Tamaño del grupo']

for ax, var, tit in zip(axes.flat, variables, titulos):
    color = '#2a78d6' if var != 'numero_estudiantes' else '#1baf7a'
    sns.histplot(df[var], bins=25, kde=True, ax=ax, color=color,
                 edgecolor='white', linewidth=0.4, alpha=0.85)
    media = df[var].mean()
    ax.axvline(media, color=INK, linestyle='--', linewidth=1)
    ax.text(media, ax.get_ylim()[1]*0.95, f' μ = {media:.2f}',
            fontsize=9, color=INK, va='top')
    ax.set_title(tit); ax.set_xlabel(''); ax.set_ylabel('Frecuencia')

fig.suptitle('Distribución de puntajes (escala 1–5) y tamaño de grupo',
             fontsize=13, fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

df[variables].describe().round(2).T

> 📌 **Insight 2 — Puntajes con asimetría izquierda y techo en 5.** Las medias rondan
> 3.7–3.85 con desviación ~0.86; la masa se concentra entre 3 y 4.5.
> **`puntaje_evaluacion` es sistemáticamente el más bajo (μ = 3.69)**: la percepción de
> justicia en la calificación es la dimensión más débil a nivel institucional — primer
> candidato para recomendaciones de mejora continua.
>
> 📌 **Insight 3 — El tamaño del grupo es ~uniforme (15–45)** y, como se confirma en § 5 y § 10,
> no guarda relación con los puntajes: descartable como palanca explicativa.

## 5 · Análisis bivariado y correlaciones

### 5.1 · Puntajes según la clase objetivo

¿Cómo se comportan las tres dimensiones evaluadas dentro de cada clase de tendencia?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, var, tit in zip(axes, COLS_PUNTAJE, ['Claridad', 'Metodología', 'Evaluación']):
    sns.boxplot(data=df, x='tendencia_desempeno', y=var, order=ORDEN_TENDENCIA,
                hue='tendencia_desempeno', palette=COLOR_TENDENCIA, legend=False,
                width=0.55, fliersize=2, linewidth=1, ax=ax)
    ax.set_title(tit); ax.set_xlabel(''); ax.set_ylabel('Puntaje (1–5)' if var == COLS_PUNTAJE[0] else '')
fig.suptitle('Distribución de puntajes por clase de tendencia', fontsize=13,
             fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

df.groupby('tendencia_desempeno')[COLS_PUNTAJE].mean().round(2).reindex(ORDEN_TENDENCIA)

### 5.2 · ¿Es la clase separable con una simple regla de umbral?

Esta es **la pregunta central del reto**: si un umbral sobre el promedio bastara para asignar
la clase, el sistema de reglas actual sería suficiente y el ML no aportaría valor.
Superponemos la densidad del **puntaje promedio** por clase para ver el solapamiento.

In [ ]:
df['puntaje_promedio'] = df[COLS_PUNTAJE].mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
for t in ORDEN_TENDENCIA:
    subset = df.loc[df['tendencia_desempeno'] == t, 'puntaje_promedio']
    sns.kdeplot(subset, ax=ax, color=COLOR_TENDENCIA[t], linewidth=2,
                fill=True, alpha=0.12, label=f'{t}  [{subset.min():.2f} – {subset.max():.2f}]')
ax.set_title('Densidad del puntaje promedio por clase — zonas de solapamiento')
ax.set_xlabel('Puntaje promedio (claridad + metodología + evaluación) / 3')
ax.set_ylabel('Densidad')
ax.legend(title='Clase  [rango observado]')
plt.tight_layout(); plt.show()

(df.groupby('tendencia_desempeno')['puntaje_promedio']
   .agg(['min', 'mean', 'max']).round(2).reindex(ORDEN_TENDENCIA))

> 📌 **Insight 4 — La clase NO es una función determinística del puntaje** (hallazgo clave
> que justifica el proyecto). Los rangos se solapan de forma importante:
> `En riesgo` llega hasta promedio **3.87** y `Mejora` comienza desde **3.43** — en la franja
> 3.4–3.9 conviven las tres clases. Un sistema de reglas por umbral se equivoca
> sistemáticamente en esa zona gris; la señal faltante está en el **comentario** (§ 8) y
> ese es el espacio de mejora del modelo de ML. En § 10.4 cuantificamos esta comparación.

### 5.3 · Correlación entre variables numéricas

In [ ]:
corr = df[COLS_PUNTAJE + ['puntaje_promedio', 'numero_estudiantes']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            vmin=-0.1, vmax=1, linewidths=1.5, linecolor='white',
            cbar_kws={'label': 'Correlación de Pearson'},
            annot_kws={'fontsize': 10}, ax=ax)
ax.set_title('Matriz de correlación (triángulo inferior)')
plt.tight_layout(); plt.show()

> 📌 **Insight 5 — Dimensiones relacionadas pero no redundantes.** Los tres puntajes
> correlacionan moderadamente entre sí (r ≈ 0.62–0.65): miden facetas distintas de un mismo
> constructo de desempeño. Ninguna alcanza r > 0.9, así que **las tres aportan información
> propia** y se conservan como features. `numero_estudiantes` no correlaciona con nada
> (r ≈ −0.03): los grupos grandes no evalúan peor.

## 6 · Análisis temporal

El reto pide **anticipar tendencias**: analizamos si existe deriva temporal a nivel
institucional y cómo se distribuye el riesgo a lo largo de los 8 semestres.

### 6.1 · Evolución institucional de los puntajes

In [ ]:
orden_sem = sorted(df['semestre'].unique())
evol = df.groupby('semestre')[COLS_PUNTAJE + ['puntaje_promedio']].mean().reindex(orden_sem)

fig, ax = plt.subplots(figsize=(10, 4.2))
estilos = {'puntaje_claridad': ('#2a78d6', 'Claridad'),
           'puntaje_metodologia': ('#1baf7a', 'Metodología'),
           'puntaje_evaluacion': ('#eda100', 'Evaluación')}
for col, (c, lbl) in estilos.items():
    ax.plot(evol.index, evol[col], marker='o', markersize=5, linewidth=2, color=c, label=lbl)
ax.plot(evol.index, evol['puntaje_promedio'], linewidth=2.5, linestyle='--',
        color=INK, label='Promedio general')
ax.set_ylim(3.3, 4.1)
ax.set_title('Evolución del puntaje medio institucional por semestre')
ax.set_ylabel('Puntaje medio (1–5)')
ax.legend(ncols=4, loc='lower center')
plt.tight_layout(); plt.show()

### 6.2 · Composición de clases por semestre

In [ ]:
comp = (pd.crosstab(df['semestre'], df['tendencia_desempeno'], normalize='index')
          .reindex(orden_sem)[ORDEN_TENDENCIA])

fig, ax = plt.subplots(figsize=(10, 4))
comp.plot(kind='bar', stacked=True, ax=ax, width=0.7,
          color=[COLOR_TENDENCIA[c] for c in comp.columns],
          edgecolor='white', linewidth=1.5)
for cont in ax.containers:
    ax.bar_label(cont, fmt=lambda v: f'{v:.0%}' if v > 0.08 else '',
                 label_type='center', color='white', fontsize=8.5, fontweight='bold')
ax.set_title('Distribución porcentual de la tendencia por semestre')
ax.set_xlabel(''); ax.set_ylabel('Proporción de evaluaciones')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(title='', ncols=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
plt.setp(ax.get_xticklabels(), rotation=0)
plt.tight_layout(); plt.show()

### 6.3 · Mapa de calor docente × semestre

Vista panorámica de los 50 docentes en el tiempo (ordenados por desempeño promedio).

In [ ]:
pivot = (df.pivot_table(index='id_docente', columns='semestre',
                         values='puntaje_promedio', aggfunc='mean')
           .reindex(columns=orden_sem))
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(9, 13))
sns.heatmap(pivot, cmap='Blues', vmin=2.8, vmax=4.6, linewidths=0.4,
            linecolor='white', cbar_kws={'label': 'Puntaje promedio', 'shrink': 0.5}, ax=ax)
ax.set_title('Puntaje promedio por docente y semestre\n(ordenado de mejor a menor desempeño histórico)')
ax.set_xlabel(''); ax.set_ylabel('')
ax.tick_params(labelsize=7.5)
plt.tight_layout(); plt.show()

> 📌 **Insight 6 — No hay deriva institucional.** El promedio general se mantiene plano
> (3.69–3.81) durante los 4 años y la mezcla de clases por semestre es estable. La "tendencia"
> **no es un fenómeno de calendario sino de docente**: la variación relevante ocurre entre
> docentes (heatmap), no entre semestres.
>
> 📌 **Insight 7 — La etiqueta no describe trayectorias.** Pese a llamarse *tendencia*, la
> etiqueta varía fila a fila dentro del mismo docente y semestre; en § 10 se confirma que el
> histórico del docente **no** la predice. Es una clasificación **transversal de cada
> evaluación**, no una serie temporal — hallazgo crítico para no diseñar el modelo equivocado.

## 7 · Análisis por asignatura

¿Existen materias estructuralmente mejor o peor evaluadas? ¿Dónde se concentra el riesgo?

In [ ]:
asig = (df.groupby('asignatura')
          .agg(puntaje_promedio=('puntaje_promedio', 'mean'),
               pct_riesgo=('tendencia_desempeno', lambda s: (s == 'En riesgo').mean()),
               n=('puntaje_promedio', 'size'))
          .sort_values('puntaje_promedio'))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

axes[0].barh(asig.index, asig['puntaje_promedio'], color='#2a78d6', height=0.6)
for i, v in enumerate(asig['puntaje_promedio']):
    axes[0].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9, color=INK2)
axes[0].set_xlim(3.4, 3.95)
axes[0].set_title('Puntaje promedio por asignatura')
axes[0].set_xlabel('Puntaje promedio (1–5)')
axes[0].grid(axis='y', visible=False)

riesgo_orden = asig.sort_values('pct_riesgo')
axes[1].barh(riesgo_orden.index, riesgo_orden['pct_riesgo'], color='#d03b3b', height=0.6)
for i, v in enumerate(riesgo_orden['pct_riesgo']):
    axes[1].text(v + 0.002, i, f'{v:.1%}', va='center', fontsize=9, color=INK2)
axes[1].set_xlim(0, 0.27)
axes[1].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[1].set_title('% de evaluaciones "En riesgo" por asignatura')
axes[1].set_xlabel('Proporción en riesgo')
axes[1].grid(axis='y', visible=False)

plt.tight_layout(); plt.show()

> 📌 **Insight 8 — La asignatura pesa poco.** El rango entre la mejor (Matemáticas Básicas,
> 3.81) y la peor (Bases de Datos, 3.70) es de apenas **0.11 puntos**, y el % en riesgo se
> mueve entre 17.5% y 21.9%. La materia por sí sola no explica el desempeño — coherente con
> su V de Cramér ≈ 0.04 (§ 10.2). Las recomendaciones deberán personalizarse **por docente**,
> no por materia.

## 8 · Análisis de comentarios (señal textual)

El comentario es la única variable no estructurada. El dataset contiene **20 comentarios
únicos** que se repiten (plantillas), lo que permite aquí una **clasificación léxica manual y
exhaustiva** de su polaridad. En producción, con texto libre real, este componente se
sustituirá por el modelo NLP del proyecto (rol de Natalia) — la arquitectura no cambia:
`comentario → señal de sentimiento → feature del modelo`.

### 8.1 · Frecuencia y polaridad de los comentarios

In [ ]:
POSITIVOS = ['Excelente profesor, muy claro.', 'Explica muy bien los temas.',
             'La clase es muy dinámica y entretenida.', 'Los talleres son muy útiles para aprender.',
             'Me inspiró a seguir aprendiendo sobre este tema.', 'Resuelve las dudas con mucha paciencia.',
             'Se nota que sabe mucho de la materia.']
NEGATIVOS = ['Califica de forma injusta.', 'Falta mucha pedagogía.',
             'La clase es muy aburrida y monótona.', 'Llega tarde a las clases frecuentemente.',
             'Los parciales no tienen nada que ver con lo visto en clase.',
             'No explica bien, me siento perdido.', 'No resuelve las dudas de los estudiantes.']

df['sentimiento'] = np.select(
    [df['comentario'].isin(POSITIVOS), df['comentario'].isin(NEGATIVOS)],
    ['positivo', 'negativo'], default='neutro')

frec = df['comentario'].value_counts()
colores = [COLOR_SENTIMIENTO[df.loc[df['comentario'] == c, 'sentimiento'].iloc[0]]
           for c in frec.index]

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(frec.index[::-1], frec.values[::-1], color=colores[::-1], height=0.65)
ax.set_title('Frecuencia de los 20 comentarios · coloreados por polaridad léxica')
ax.set_xlabel('Número de evaluaciones')
ax.grid(axis='y', visible=False)
ax.tick_params(axis='y', labelsize=8.5)
handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR_SENTIMIENTO[s]) for s in
           ['positivo', 'neutro', 'negativo']]
ax.legend(handles, ['Positivo', 'Neutro', 'Negativo'], loc='lower right')
plt.tight_layout(); plt.show()

print(df['sentimiento'].value_counts(normalize=True).round(3))

### 8.2 · Relación sentimiento ↔ clase objetivo

In [ ]:
cruce = (pd.crosstab(df['sentimiento'], df['tendencia_desempeno'], normalize='index')
           [ORDEN_TENDENCIA].reindex(['positivo', 'neutro', 'negativo']))

fig, ax = plt.subplots(figsize=(9, 3.6))
cruce.plot(kind='barh', stacked=True, ax=ax, width=0.6,
           color=[COLOR_TENDENCIA[c] for c in cruce.columns],
           edgecolor='white', linewidth=1.5)
for cont in ax.containers:
    ax.bar_label(cont, fmt=lambda v: f'{v:.0%}' if v > 0.05 else '',
                 label_type='center', color='white', fontsize=9, fontweight='bold')
ax.set_title('Distribución de la clase objetivo según la polaridad del comentario')
ax.set_xlabel('Proporción de evaluaciones'); ax.set_ylabel('')
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(ncols=3, loc='upper center', bbox_to_anchor=(0.5, -0.18))
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

df.groupby('sentimiento')['puntaje_promedio'].agg(['mean', 'count']).round(2)

> 📌 **Insight 9 — El comentario contiene una restricción determinística de gran valor.**
> * Un comentario **negativo jamás coexiste con la clase `Mejora`** (0 casos en 3,000).
> * Un comentario **positivo jamás coexiste con `En riesgo`** (0 casos).
> * El puntaje promedio escala con la polaridad: 3.35 (negativo) → 3.73 (neutro) → 4.13 (positivo).
>
> Esta es exactamente la señal que el sistema de reglas actual (basado solo en puntajes)
> **no captura**, y explica gran parte de la "zona gris" del Insight 4. **El análisis de
> sentimiento del comentario es la feature complementaria más valiosa del proyecto** y valida
> la línea de trabajo NLP del plan de acción.

## 9 · Perfil por docente

La unidad de acción del negocio es el **docente**: las recomendaciones de mejora continua se
entregan por persona. Construimos su perfil agregado y detectamos los casos extremos.

In [ ]:
perfil = (df.groupby('id_docente')
            .agg(puntaje_promedio=('puntaje_promedio', 'mean'),
                 pct_riesgo=('tendencia_desempeno', lambda s: (s == 'En riesgo').mean()),
                 pct_mejora=('tendencia_desempeno', lambda s: (s == 'Mejora').mean()),
                 n_evaluaciones=('puntaje_promedio', 'size'),
                 sentimiento_medio=('sentimiento', lambda s: s.map(
                     {'negativo': -1, 'neutro': 0, 'positivo': 1}).mean())))

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(perfil['puntaje_promedio'], perfil['pct_riesgo'],
                s=perfil['n_evaluaciones'] * 2.2, c='#2a78d6', alpha=0.65,
                edgecolors='white', linewidths=1)
ax.axhline(perfil['pct_riesgo'].mean(), color=MUTED, linestyle='--', linewidth=0.9)
ax.axvline(perfil['puntaje_promedio'].mean(), color=MUTED, linestyle='--', linewidth=0.9)

destacados = pd.concat([perfil.nsmallest(3, 'puntaje_promedio'),
                        perfil.nlargest(3, 'puntaje_promedio')])
for nombre, fila in destacados.iterrows():
    ax.annotate(nombre, (fila['puntaje_promedio'], fila['pct_riesgo']),
                textcoords='offset points', xytext=(7, 5), fontsize=8.5, color=INK)

ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title('Mapa de docentes: desempeño promedio vs. exposición al riesgo\n(tamaño ∝ número de evaluaciones)')
ax.set_xlabel('Puntaje promedio histórico (1–5)')
ax.set_ylabel('% de evaluaciones "En riesgo"')
plt.tight_layout(); plt.show()

print('— Docentes con mayor exposición al riesgo —')
print(perfil.nlargest(5, 'pct_riesgo').round(3).to_string())
print()
print('— Docentes de mejor desempeño —')
print(perfil.nlargest(5, 'puntaje_promedio').round(3).to_string())

> 📌 **Insight 10 — Heterogeneidad real entre docentes, pero con rangos acotados.** El
> promedio histórico por docente va de **3.56 (Docente_4)** a **4.00 (Docente_18)** y la
> exposición al riesgo de **10.5% a 29.2%** — casi 3× entre extremos. El cuadrante
> inferior-derecho del mapa (alto puntaje, bajo riesgo) define el *benchmark* interno;
> el superior-izquierdo (Docente_4, Docente_50, Docente_16) concentra la prioridad de
> acompañamiento pedagógico. Este mapa es el embrión del **tablero de recomendaciones**
> del producto final.

## 10 · SCRUM-8 · Significancia de features para la variable objetivo

Cuantificamos formalmente qué variables predicen `tendencia_desempeno`, con **cuatro
métodos complementarios** (dos estadísticos, dos de ML) para que la conclusión no dependa
de un solo criterio:

| Método | Qué mide | Aplica a |
|---|---|---|
| Kruskal-Wallis | ¿Difiere la distribución de la variable entre las 3 clases? (no paramétrico) | numéricas |
| V de Cramér | Fuerza de asociación con la clase (0 = nada, 1 = perfecta) | categóricas |
| Información mutua | Dependencia no lineal con la clase | todas |
| Importancia de permutación | Caída real de macro-F1 al aleatorizar la feature en un modelo entrenado | todas |

### 10.1 · Ingeniería de features candidatas

Además de las variables originales, construimos candidatas alineadas con la hipótesis
"temporal" del reto — si la historia del docente ayuda a anticipar su estado:

* `puntaje_promedio` — media de las 3 dimensiones (resumen del sistema de reglas actual).
* `sentimiento_score` — polaridad del comentario codificada (−1, 0, +1).
* `hist_prom_previo` — promedio **histórico acumulado** del docente hasta el semestre anterior (sin fuga de información).
* `delta_semestre` — cambio del promedio del docente vs. su semestre anterior.

In [ ]:
df['sentimiento_score'] = df['sentimiento'].map({'negativo': -1, 'neutro': 0, 'positivo': 1})

# --- Features temporales por docente (solo con información PASADA: shift antes de acumular) ---
doc_sem = (df.groupby(['id_docente', 'semestre'])['puntaje_promedio']
             .mean().rename('prom_docente_sem').reset_index()
             .sort_values(['id_docente', 'semestre']))
doc_sem['prom_sem_anterior'] = doc_sem.groupby('id_docente')['prom_docente_sem'].shift(1)
doc_sem['hist_prom_previo'] = (doc_sem.groupby('id_docente')['prom_docente_sem']
                                      .transform(lambda s: s.shift(1).expanding().mean()))
doc_sem['delta_semestre'] = doc_sem['prom_docente_sem'] - doc_sem['prom_sem_anterior']

df = df.merge(doc_sem[['id_docente', 'semestre', 'hist_prom_previo', 'delta_semestre']],
              on=['id_docente', 'semestre'], how='left')

FEATURES_NUM = COLS_PUNTAJE + ['puntaje_promedio', 'numero_estudiantes',
                               'sentimiento_score', 'hist_prom_previo', 'delta_semestre']
print(f'Features numéricas candidatas: {len(FEATURES_NUM)}')
print(f'Filas con historial disponible (semestre ≥ segundo del docente): '
      f'{df["hist_prom_previo"].notna().sum():,} de {len(df):,}')

### 10.2 · Pruebas estadísticas

In [ ]:
# --- Kruskal-Wallis: numéricas vs clase ---
resultados = []
for feat in FEATURES_NUM:
    datos = df.dropna(subset=[feat])
    grupos = [datos.loc[datos['tendencia_desempeno'] == t, feat] for t in ORDEN_TENDENCIA]
    H, p = stats.kruskal(*grupos)
    resultados.append({'feature': feat, 'H de Kruskal-Wallis': round(H, 1),
                       'p-valor': f'{p:.2e}', 'significativa (α=0.01)': '✅' if p < 0.01 else '❌'})
pd.DataFrame(resultados).sort_values('H de Kruskal-Wallis', ascending=False).reset_index(drop=True)

In [ ]:
# --- V de Cramér: categóricas vs clase ---
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tabla)[0]
    n = tabla.values.sum()
    return np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

categoricas = ['sentimiento', 'id_docente', 'semestre', 'asignatura']
pd.DataFrame({
    'feature': categoricas,
    'V de Cramér': [round(cramers_v(df[c], df['tendencia_desempeno']), 3) for c in categoricas],
}).sort_values('V de Cramér', ascending=False).reset_index(drop=True)

### 10.3 · Información mutua

In [ ]:
from sklearn.feature_selection import mutual_info_classif

datos_mi = df.dropna(subset=FEATURES_NUM).copy()
datos_mi['asignatura_cod'] = datos_mi['asignatura'].astype('category').cat.codes
X_mi = datos_mi[FEATURES_NUM + ['asignatura_cod']]
y_mi = datos_mi['tendencia_desempeno']

mi = pd.Series(mutual_info_classif(X_mi, y_mi, random_state=RANDOM_STATE),
               index=X_mi.columns).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(mi.index, mi.values, color='#2a78d6', height=0.6)
for i, v in enumerate(mi.values):
    ax.text(v + 0.008, i, f'{v:.3f}', va='center', fontsize=9, color=INK2)
ax.set_xlim(0, mi.max() * 1.18)
ax.set_title('Información mutua de cada feature con la clase objetivo')
ax.set_xlabel('Información mutua (nats)')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

### 10.4 · Validación con modelo: ¿cuánto aporta cada feature en la práctica?

Entrenamos un **RandomForest de referencia** (no es el modelo final, es un instrumento de
medición) y lo comparamos contra el **mejor sistema de reglas posible** — dos umbrales sobre
`puntaje_promedio` optimizados por búsqueda exhaustiva. Esta comparación operacionaliza la
premisa del reto: *reglas vs. aprendizaje automático*.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.inspection import permutation_importance

X = X_mi
y = y_mi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# --- Referencia 1: mejor sistema de REGLAS (2 umbrales sobre puntaje_promedio) ---
def aplicar_regla(prom, t_bajo, t_alto):
    return np.where(prom < t_bajo, 'En riesgo', np.where(prom > t_alto, 'Mejora', 'Estable'))

mejor = (0, None)
for t_bajo in np.arange(2.6, 3.9, 0.05):
    for t_alto in np.arange(t_bajo + 0.1, 4.8, 0.05):
        f1 = f1_score(y_train, aplicar_regla(X_train['puntaje_promedio'], t_bajo, t_alto),
                      average='macro')
        if f1 > mejor[0]:
            mejor = (f1, (t_bajo, t_alto))
t_bajo, t_alto = mejor[1]
pred_regla = aplicar_regla(X_test['puntaje_promedio'], t_bajo, t_alto)

# --- Referencia 2: RandomForest con todas las features ---
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                            random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print(f'Sistema de reglas óptimo  (umbral bajo={t_bajo:.2f}, alto={t_alto:.2f}):')
print(f'   accuracy = {accuracy_score(y_test, pred_regla):.3f} | '
      f'macro-F1 = {f1_score(y_test, pred_regla, average="macro"):.3f}')
print(f'RandomForest (todas las features):')
print(f'   accuracy = {accuracy_score(y_test, pred_rf):.3f} | '
      f'macro-F1 = {f1_score(y_test, pred_rf, average="macro"):.3f}')
print()
print(classification_report(y_test, pred_rf))

In [ ]:
# --- Importancia de permutación (sobre el conjunto de prueba, métrica macro-F1) ---
perm = permutation_importance(rf, X_test, y_test, n_repeats=15,
                              random_state=RANDOM_STATE, scoring='f1_macro')
imp = (pd.Series(perm.importances_mean, index=X.columns)
         .sort_values())

fig, ax = plt.subplots(figsize=(9, 4.5))
colores_imp = ['#2a78d6' if v > 0.005 else '#c3c2b7' for v in imp.values]
ax.barh(imp.index, imp.values, color=colores_imp, height=0.6)
for i, v in enumerate(imp.values):
    ax.text(max(v, 0) + 0.004, i, f'{v:+.3f}', va='center', fontsize=9, color=INK2)
ax.set_xlim(min(imp.min(), 0) - 0.01, imp.max() * 1.18)
ax.set_title('Importancia de permutación (caída de macro-F1 al aleatorizar cada feature)')
ax.set_xlabel('Δ macro-F1')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

### 10.5 · Ranking consolidado de features (conclusión SCRUM-8)

| # | Feature | Kruskal-Wallis | Inf. mutua | Imp. permutación | Veredicto |
|---|---|---|---|---|---|
| 1 | `puntaje_promedio` | H≈2337 ✅ | ≈0.80 | ≈+0.48 | **Feature dominante** — resume el 80% de la señal |
| 2 | `sentimiento_score` (comentario) | H≈642 ✅ | ≈0.21 | ≈+0.08 | **Segunda señal y la única complementaria** — aporta información que los puntajes no tienen |
| 3–5 | `puntaje_metodologia` / `claridad` / `evaluacion` | H≈1700–1800 ✅ | ≈0.46–0.48 | ≈+0.01–0.03 | Individualmente fuertes pero **redundantes** con el promedio; conservar para interpretabilidad por dimensión |
| 6 | `delta_semestre` | H≈141 ✅ | ≈0.00 | ≈0 | Estadísticamente significativa pero sin aporte predictivo marginal |
| 7 | `id_docente` | V Cramér ≈ 0.12 | — | — | Señal débil de perfil; útil para el tablero, no para el modelo transversal |
| 8+ | `hist_prom_previo`, `numero_estudiantes`, `asignatura`, `semestre` | n.s. / V≈0.04–0.05 | ≈0.00 | ≈0 | **Sin poder predictivo** sobre esta etiqueta |

> 📌 **Insight 11 (conclusión central de SCRUM-8).** La etiqueta se explica por
> **el desempeño puntual de la evaluación (puntajes) + la polaridad del comentario**, y NO por
> la historia del docente, la asignatura, el semestre ni el tamaño del grupo. Dos consecuencias
> de diseño:
> 1. El modelo de la siguiente fase debe ser un **clasificador transversal por evaluación**
>    cuyo eje diferenciador es el **procesamiento del comentario (NLP)** — con texto libre real,
>    reemplazar la codificación léxica por embeddings o un modelo de sentimiento en español
>    (p. ej. BETO/RoBERTuito fine-tuned), que es donde el ML puede superar con claridad a las reglas.
> 2. El mejor sistema de reglas alcanza ≈0.90 de macro-F1 usando solo el puntaje; el valor
>    agregado del ML está en la **zona gris de solapamiento** (Insight 4), en producir
>    **probabilidades calibradas por clase** (priorización de docentes) y en absorber la señal
>    textual — no en una mejora masiva de accuracy sobre datos sintéticos.

## 11 · Conclusiones, limitaciones y próximos pasos

### 🧭 Síntesis de insights

| # | Insight | Implicación para la solución |
|---|---|---|
| 1 | Clases desbalanceadas (`En riesgo` = 20%) | Métrica macro-F1 + recall de riesgo; `class_weight` |
| 2 | `puntaje_evaluacion` es la dimensión más débil (μ=3.69) | Primera recomendación institucional de mejora |
| 3 | Tamaño de grupo sin relación con desempeño | Descartar como feature |
| 4 | Rangos de clase solapados (franja gris 3.4–3.9) | Las reglas fallan ahí: espacio de valor del ML |
| 5 | Puntajes correlacionados (0.62–0.65) pero no redundantes | Conservar las 3 dimensiones |
| 6 | Sin deriva temporal institucional (promedio plano 4 años) | El problema es transversal, no de serie de tiempo |
| 7 | La etiqueta no depende del historial del docente | No modelar como forecasting; clasificador por evaluación |
| 8 | La asignatura pesa poco (Δ máx 0.11 pts) | Recomendaciones por docente, no por materia |
| 9 | Comentario negativo ⇒ nunca `Mejora`; positivo ⇒ nunca `En riesgo` | **El NLP del comentario es la feature complementaria clave** |
| 10 | Riesgo por docente varía 10.5% → 29.2% (≈3×) | Mapa de priorización de acompañamiento |
| 11 | Señal = puntajes + sentimiento; resto sin aporte | Arquitectura del modelo definida (§ 10.5) |

### ⚠️ Limitaciones del dataset (para el informe técnico y el capítulo de ética)

* **Origen sintético evidente**: 20 comentarios plantilla, 50 docentes dictando las 7
  asignaturas, mezcla de clases estable por construcción. Los resultados **validan la
  metodología**, pero las métricas absolutas no son extrapolables a datos reales.
* La etiqueta `tendencia_desempeno` viene del sistema actual basado en reglas: el modelo
  aprende en parte a *imitar* ese sistema. Con datos reales convendrá redefinir la etiqueta
  con criterio institucional (p. ej. trayectoria real inter-semestral del docente).
* **Ética**: datos de desempeño de personas → anonimización estricta (ya presente), uso del
  modelo como herramienta de *acompañamiento* y no de sanción, y monitoreo de sesgos por
  asignatura/antigüedad cuando existan esos atributos.

### 🚀 Próximos pasos (fase de modelado)

1. **Pipeline de features**: puntajes + score de sentimiento (léxico ahora, NLP después).
2. **Modelos candidatos**: Regresión Logística (línea base interpretable) → RandomForest /
   Gradient Boosting → comparación formal contra el sistema de reglas optimizado (§ 10.4).
3. **NLP en español** sobre comentarios reales: embeddings o transformer fine-tuned
   (línea de trabajo de Natalia).
4. **Validación**: estratificada + análisis específico de la franja gris (3.4–3.9) donde el
   ML debe demostrar su ventaja; calibración de probabilidades para priorizar docentes.
5. **Producto**: tablero por docente (mapa del § 9) con probabilidad de riesgo,
   dimensiones débiles y recomendaciones generadas.

---
*Notebook desarrollado por el equipo **Tech Tailors** · Samsung Innovation Campus 2025 ·
Reto Analítica Educativa (Universidad del Rosario).*